# PhiSpy

### Detecting golden-ratio structure in photos, with actual statistics

When I was a kid, I watched an old Disney educational short about math that spent a few
minutes on the golden ratio — the idea that a specific proportion, roughly 1.618, shows up
again and again in shells, flowers, art, and architecture, and that things built around it
tend to look more pleasing. It stuck with me for years as one of those "math is secretly
everywhere" facts.

As an adult who now writes code for a living, the claim started to bug me in a different
way: *how would you actually check that?* A lot of the golden-ratio-in-art claims floating
around online amount to "I drew a spiral over this painting and it kind of fit," which is
not exactly a rigorous test. So this notebook is my attempt to build an actual measurement
pipeline for it: detect structurally significant points in an image, look for pairs of
distances between those points whose ratio is close to phi, and then — the part most
pop-science treatments skip — check whether that count of "phi matches" is actually higher
than you'd expect from pure chance, given how many points and how many possible pairings
there are.

The result is a single number I'm calling the **aesthetic index**: a z-score describing how
far an image's golden-ratio structure deviates from a random baseline. It is *not* a beauty
score, and section 9 spends real time on what this project does and doesn't prove.

## 0. Setup

In [ ]:
import math
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)

## 1. What is the golden ratio?

The golden ratio, usually written as the Greek letter phi ($\varphi$), is the positive
solution to:

$$\varphi = 1 + \frac{1}{\varphi}$$

which works out to the closed form:

$$\varphi = \frac{1 + \sqrt{5}}{2} \approx 1.6180339887\ldots$$

**Self-similarity.** If you take a rectangle whose side lengths are in the ratio $\varphi:1$
(a "golden rectangle") and cut off the largest possible square, the leftover rectangle is
*also* a golden rectangle — just smaller. You can repeat this forever, and if you inscribe a
quarter-circle arc in each square, you get the golden spiral. This self-similar,
repeat-forever property is the main reason phi shows up in growth patterns in nature (shell
chambers, seed heads, branching) — it's a ratio that's compatible with a shape scaling up
while keeping its proportions.

**Fibonacci connection.** The Fibonacci sequence (1, 1, 2, 3, 5, 8, 13, 21, ...), where each
number is the sum of the two before it, has the property that the ratio of consecutive terms
converges to phi as the sequence grows:

$$\lim_{n \to \infty} \frac{F_{n+1}}{F_n} = \varphi$$

That's why Fibonacci spirals and golden spirals are visually similar — the Fibonacci ratio is
a discrete approximation that converges to the true, continuous golden ratio.

Below, we generate a golden rectangle subdivision and the corresponding golden spiral
directly with matplotlib (no downloaded reference image) to make the geometry concrete.

In [ ]:
PHI = (1 + math.sqrt(5)) / 2
print(f'phi = {PHI:.10f}')

# Sanity-check the defining property phi = 1 + 1/phi
print(f'1 + 1/phi = {1 + 1 / PHI:.10f}')

# Sanity-check the Fibonacci convergence
fib = [1, 1]
for _ in range(20):
    fib.append(fib[-1] + fib[-2])
print(f'F(21)/F(20) = {fib[-1] / fib[-2]:.10f}  (approaches phi)')

In [ ]:
def plot_golden_rectangle_and_spiral(n_squares=8, ax=None):
    """Draws a golden-rectangle square subdivision plus the golden spiral inscribed in it."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))

    # Start with a golden rectangle: width = phi, height = 1, anchored at (0, 0).
    x, y, w, h = 0.0, 0.0, PHI, 1.0

    # Which side we cut the square from alternates as we spiral inward: right, top, left, bottom, ...
    directions = ['right', 'top', 'left', 'bottom']

    for i in range(n_squares):
        direction = directions[i % 4]
        side = min(w, h)  # the square we cut off has side length = shorter edge of the rectangle

        if direction == 'right':
            square_origin = (x + w - side, y)
            center, r, start_angle = (x + w - side, y + side), side, 180
            x, w = x, w - side
        elif direction == 'top':
            square_origin = (x, y + h - side)
            center, r, start_angle = (x, y + h - side), side, 270
            y, h = y, h - side
        elif direction == 'left':
            square_origin = (x, y)
            center, r, start_angle = (x + side, y), side, 0
            x, w = x + side, w - side
        else:  # bottom
            square_origin = (x, y)
            center, r, start_angle = (x + side, y + side), side, 90
            y, h = y + side, h - side

        # Draw the square for this iteration
        ax.add_patch(plt.Rectangle(square_origin, side, side, fill=False,
                                    edgecolor='#888888', linewidth=1))

        # Draw the quarter-circle arc that approximates the golden spiral inside this square
        theta = np.linspace(start_angle, start_angle + 90, 50) * np.pi / 180
        arc_x = center[0] + r * np.cos(theta)
        arc_y = center[1] + r * np.sin(theta)
        ax.plot(arc_x, arc_y, color='#d4820a', linewidth=2)

    ax.set_xlim(-0.05, PHI + 0.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.set_title(f'Golden rectangle subdivision + spiral ($\\varphi \\approx {PHI:.4f}$)')
    ax.axis('off')
    return ax

plot_golden_rectangle_and_spiral()
plt.show()

## 2. How computer vision finds candidate points

To test for golden-ratio structure in a *photo*, we first need a set of points that plausibly
correspond to "structurally interesting" locations — corners, edges, and high-contrast
features — rather than every single pixel.

**Canny edge detection** finds pixels where intensity changes sharply (likely object
boundaries) by looking at the image gradient, thinning the result down to single-pixel-wide
edges, and then keeping only edges strong and connected enough to survive two intensity
thresholds (hysteresis thresholding). It's great for visualizing *where* structure is, but an
edge map is thousands of points — too many, and too redundant, for pairwise-distance analysis.

**Shi-Tomasi corner detection** (`cv2.goodFeaturesToTrack`, an improvement on the earlier
Harris corner detector) instead looks for points where the local image gradient varies in
*two* directions at once — i.e. actual corners, not just edges. It does this by computing,
for a small window around each pixel, a structure tensor (a 2x2 matrix built from the
local gradients) and checking its eigenvalues: if both eigenvalues are large, the window looks
like a corner in every direction; if only one is large, it's just an edge; if both are small,
it's a flat region. Harris scores corners with a function of those eigenvalues; Shi-Tomasi
(used here) directly uses the smaller eigenvalue, which tends to give more stable corners for
tracking. This gives us a much smaller, more meaningful set of candidate points — exactly
what we want for the pairwise-ratio analysis in the next sections.

In [ ]:
def detect_candidate_points(image, max_corners=80, quality_level=0.01, min_distance=15):
    """Detects salient (Shi-Tomasi) corner points in an image.

    Parameters
    ----------
    image : str or np.ndarray
        Path to an image file, or an already-loaded BGR image array (as returned by cv2.imread).
    max_corners : int
        Maximum number of corners to return. Kept modest by default since the pairwise-ratio
        analysis below is roughly O(n^4) in the number of points.
    quality_level : float
        Minimum accepted corner quality, as a fraction of the best corner's quality score.
        Lower values keep more (weaker) corners.
    min_distance : int
        Minimum pixel distance enforced between returned corners, to avoid clusters of
        near-duplicate points in one small region.

    Returns
    -------
    points : np.ndarray of shape (N, 2)
        (x, y) pixel coordinates of detected points.
    gray : np.ndarray
        The grayscale image the detector ran on (handy for visualization/debugging).
    """
    if isinstance(image, str):
        img = cv2.imread(image)
        if img is None:
            raise FileNotFoundError(f'Could not read image at {image!r}')
    else:
        img = image

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    corners = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=max_corners,
        qualityLevel=quality_level,
        minDistance=min_distance,
    )

    if corners is None:
        return np.empty((0, 2), dtype=np.float64), gray

    points = corners.reshape(-1, 2).astype(np.float64)
    return points, gray

In [ ]:
def visualize_points(image, points, title=None, point_color=(0, 255, 0), radius=6):
    """Overlays detected points on an image and displays it with matplotlib.

    Parameters
    ----------
    image : str or np.ndarray
        Path to an image file, or a BGR image array.
    points : np.ndarray of shape (N, 2)
        (x, y) coordinates to draw, e.g. from detect_candidate_points().
    title : str, optional
    point_color : tuple
        BGR color for the markers (OpenCV convention).
    radius : int
        Marker radius in pixels.
    """
    if isinstance(image, str):
        img = cv2.imread(image)
        if img is None:
            raise FileNotFoundError(f'Could not read image at {image!r}')
    else:
        img = image.copy()

    for x, y in points:
        cv2.circle(img, (int(round(x)), int(round(y))), radius, point_color, thickness=2)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(f'{title} ({len(points)} points)')
    plt.show()

## 3. The problem: raw match counts are (almost) meaningless

Here's the issue that this whole project is actually about.

Say we detect $n$ candidate points in an image. The number of *pairs of points* is
$\binom{n}{2}$, so the number of *distances* between points is $\binom{n}{2}$, and the number
of *pairs of distances* we could compare to each other (and thus check for a ratio near phi)
is roughly $\binom{\binom{n}{2}}{2}$ — which grows like $n^4$. For just 40 points, that's
already on the order of a million distance-pairs to check.

If we're scanning a million-plus ratios and asking "is this close to 1.618?" with any
reasonable tolerance (say, within 2%), **we will find a substantial number of matches in
literally any scatter of points — including a scatter with no structure whatsoever, like
random noise.** This is the same statistical trap as p-hacking: run enough comparisons and
you will find "significant" results by chance alone.

This is exactly the flaw in a lot of casual golden-ratio-in-art claims: someone finds *a*
pair of distances in a painting that's close to phi, declares the artist secretly used the
golden ratio, and never asks the obvious follow-up question — *how many phi-ish ratios would
I find in a completely random set of points of the same size?* If the answer is "about the
same number," the "discovery" is noise, not signal.

So before we can say anything meaningful about an image, we need to answer that follow-up
question directly, with a random baseline. That's section 4.

## 4. The solution: compare against a random baseline

The fix is conceptually simple, and it's the standard move for this kind of "is my count
surprising" question: **simulate the null hypothesis.**

1. Take the real image's detected points, and count how many pairs of distances have a ratio
   within some tolerance of phi. Call this `observed`.
2. Generate a set of the *same number* of points, scattered uniformly at random over an area
   the same size as the image. Run the *identical* matching procedure on them, and count the
   matches.
3. Repeat step 2 many times (e.g. 200–1000 trials) to build a distribution of "how many phi
   matches happen by pure chance" for that many random points in that size of image.
4. Compare `observed` to that distribution using a z-score:

$$z = \frac{\text{observed} - \text{mean(baseline)}}{\text{std(baseline)}}$$

A z-score near 0 means the image's phi-matches look just like chance. A z-score well above 0
(roughly, above 2) means the image has meaningfully *more* phi-ratio structure than a random
scatter of the same number of points would produce — which is what we'll call the
**aesthetic index**.

This doesn't "prove" an artist intended the golden ratio, and it doesn't measure beauty — it
answers a narrower, honest question: *is this specific kind of geometric structure present
more than chance would predict?* Section 9 discusses the limits of that claim in more depth.

## 5. Core functions

Now we implement the pipeline described above: `pairwise_ratio_matches()` for the matching
logic, `random_baseline()` to simulate the null distribution, `aesthetic_index()` to tie
everything together into a single z-score per image, and `print_result()` as a small
formatting helper.

In [ ]:
def pairwise_ratio_matches(points, target_ratio=PHI, tolerance=0.02):
    """Counts pairs of pairwise distances whose ratio is close to target_ratio.

    For a set of N points, this looks at every distance between two points (there are
    N*(N-1)/2 of these), and then at every pair of those distances, checking whether the
    larger divided by the smaller falls within `tolerance` (relative) of `target_ratio`.

    Note: this is O(N^4) overall (O(N^2) distances, compared pairwise), so keep N modest
    (detect_candidate_points defaults to max_corners=80 for this reason).

    Parameters
    ----------
    points : np.ndarray of shape (N, 2)
    target_ratio : float
        The ratio we're looking for (phi, by default).
    tolerance : float
        Relative tolerance: a ratio r counts as a match if
        abs(r - target_ratio) <= tolerance * target_ratio.

    Returns
    -------
    int : number of distance-pairs whose ratio matched target_ratio within tolerance.
    """
    n = len(points)
    if n < 3:
        return 0

    # All pairwise distances, as a flat 1D array (upper triangle of the distance matrix).
    diffs = points[:, None, :] - points[None, :, :]
    dist_matrix = np.sqrt((diffs ** 2).sum(axis=-1))
    iu = np.triu_indices(n, k=1)
    distances = dist_matrix[iu]
    distances = distances[distances > 1e-9]  # drop (near-)zero distances between duplicate points

    if len(distances) < 2:
        return 0

    # Compare every pair of distances at once via broadcasting: ratio[i, j] = distances[i] / distances[j].
    # Only keep the upper triangle (i < j) so each unordered pair of distances is counted once,
    # and always divide the larger by the smaller so ratios are >= 1.
    d = distances
    m = len(d)
    jm, jn = np.triu_indices(m, k=1)
    larger = np.maximum(d[jm], d[jn])
    smaller = np.minimum(d[jm], d[jn])
    ratios = larger / smaller

    matches = np.abs(ratios - target_ratio) <= (tolerance * target_ratio)
    return int(matches.sum())

In [ ]:
def random_baseline(num_points, image_shape, target_ratio=PHI, tolerance=0.02,
                     trials=200, seed=None):
    """Simulates the null distribution of phi-matches for randomly scattered points.

    Parameters
    ----------
    num_points : int
        How many points to scatter per trial (should match the real image's point count).
    image_shape : tuple
        (height, width) of the area to scatter points over — typically the analyzed image's
        shape, so the random points live in a comparably-sized and comparably-shaped area.
    target_ratio, tolerance : see pairwise_ratio_matches().
    trials : int
        Number of random point sets to simulate. 200–1000 is a reasonable range; more trials
        gives a more stable baseline mean/std at the cost of runtime.
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    np.ndarray of shape (trials,) : match counts from each simulated trial.
    """
    rng = np.random.default_rng(seed)
    height, width = image_shape[:2]
    counts = np.empty(trials, dtype=int)

    for i in range(trials):
        random_points = np.column_stack([
            rng.uniform(0, width, num_points),
            rng.uniform(0, height, num_points),
        ])
        counts[i] = pairwise_ratio_matches(random_points, target_ratio=target_ratio,
                                            tolerance=tolerance)

    return counts

In [ ]:
def aesthetic_index(image, max_corners=80, quality_level=0.01, min_distance=15,
                     target_ratio=PHI, tolerance=0.02, trials=200, seed=None):
    """Runs the full PhiSpy pipeline on one image and returns a dict of results.

    Detects candidate points, counts observed phi-ratio matches, simulates a random baseline
    with the same number of points over the same image area, and computes the z-score
    ('aesthetic index') of the observed count relative to that baseline.

    Parameters
    ----------
    image : str or np.ndarray
        Path to an image file, or a pre-loaded BGR image array.
    max_corners, quality_level, min_distance : passed to detect_candidate_points().
    target_ratio, tolerance : passed to pairwise_ratio_matches() / random_baseline().
    trials : passed to random_baseline().
    seed : int, optional, for reproducible baselines.

    Returns
    -------
    dict with keys:
        'points', 'gray_image' : detected points and the grayscale image used
        'num_points' : int
        'observed' : observed phi-match count
        'baseline_counts' : np.ndarray of per-trial baseline counts
        'baseline_mean', 'baseline_std' : summary stats of the baseline
        'z_score' : the aesthetic index
    """
    points, gray = detect_candidate_points(
        image, max_corners=max_corners, quality_level=quality_level, min_distance=min_distance
    )
    num_points = len(points)

    observed = pairwise_ratio_matches(points, target_ratio=target_ratio, tolerance=tolerance)
    baseline_counts = random_baseline(
        num_points, gray.shape, target_ratio=target_ratio, tolerance=tolerance,
        trials=trials, seed=seed,
    )

    baseline_mean = float(baseline_counts.mean())
    baseline_std = float(baseline_counts.std())

    if baseline_std > 0:
        z_score = (observed - baseline_mean) / baseline_std
    else:
        # No variance in the baseline (e.g. it's always 0) — any nonzero observed count
        # is trivially 'infinitely' surprising, otherwise there's nothing to distinguish.
        z_score = 0.0 if observed == baseline_mean else float('inf') * np.sign(observed - baseline_mean)

    return {
        'points': points,
        'gray_image': gray,
        'num_points': num_points,
        'observed': observed,
        'baseline_counts': baseline_counts,
        'baseline_mean': baseline_mean,
        'baseline_std': baseline_std,
        'z_score': z_score,
    }

In [ ]:
def print_result(result, name=None):
    """Pretty-prints an aesthetic_index() result dict."""
    label = f'{name}: ' if name else ''
    print(f'{label}{result["num_points"]} points detected')
    print(f'  Observed phi-matches:   {result["observed"]}')
    print(f'  Random baseline:        mean={result["baseline_mean"]:.2f}, '
          f'std={result["baseline_std"]:.2f}')
    print(f'  Aesthetic index (z):    {result["z_score"]:.3f}')

## 6. Testing

Below are one section per category. Each is set up with a placeholder path, commented out.
Drop your own photos (or public-domain / openly-licensed images — please don't use
copyrighted images you don't have rights to) into the matching `images/<category>/` folder,
then uncomment and update the path to try it.

The **control** section uses a low-structure image (e.g. a scribble, or scattered dots) as a
sanity check: if the pipeline is working correctly, a genuinely unstructured control image
should produce an aesthetic index close to 0.

### Paintings

In [ ]:
# image_path = 'images/paintings/your_painting.jpg'
# painting_result = aesthetic_index(image_path, seed=42)
# print_result(painting_result, name='your_painting.jpg')
# points, gray = painting_result['points'], painting_result['gray_image']
# visualize_points(image_path, points, title='Paintings')

### Architecture

In [ ]:
# image_path = 'images/architecture/your_building.jpg'
# architecture_result = aesthetic_index(image_path, seed=42)
# print_result(architecture_result, name='your_building.jpg')
# points, gray = architecture_result['points'], architecture_result['gray_image']
# visualize_points(image_path, points, title='Architecture')

### Nature

In [ ]:
# image_path = 'images/nature/your_landscape.jpg'
# nature_result = aesthetic_index(image_path, seed=42)
# print_result(nature_result, name='your_landscape.jpg')
# points, gray = nature_result['points'], nature_result['gray_image']
# visualize_points(image_path, points, title='Nature')

### Control

A deliberately low-structure image — a scribble, static, or randomly scattered dots — used
to sanity-check the pipeline. A well-behaved control should land close to an aesthetic index
of 0, since it has no intentional geometric structure for the detector to find.

In [ ]:
# image_path = 'images/control/scribble.jpg'
# control_result = aesthetic_index(image_path, seed=42)
# print_result(control_result, name='scribble.jpg')
# points, gray = control_result['points'], control_result['gray_image']
# visualize_points(image_path, points, title='Control')

## 7. Results comparison

Once you've run the sections above on your own images, collect the results here into a
dataframe and compare aesthetic index by category. This cell assumes the variable names from
section 6 (`painting_result`, `architecture_result`, `nature_result`, `control_result`); adjust
if you tested multiple images per category (e.g. build a list of dicts, one per image, each
tagged with its category, instead of one variable per category).

In [ ]:
# results_table = pd.DataFrame([
#     {'category': 'Paintings',    'image': 'your_painting.jpg', 'aesthetic_index': painting_result['z_score']},
#     {'category': 'Architecture', 'image': 'your_building.jpg', 'aesthetic_index': architecture_result['z_score']},
#     {'category': 'Nature',       'image': 'your_landscape.jpg', 'aesthetic_index': nature_result['z_score']},
#     {'category': 'Control',      'image': 'scribble.jpg',      'aesthetic_index': control_result['z_score']},
# ])
# results_table

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 5))
# colors = ['#d4820a' if c != 'Control' else '#888888' for c in results_table['category']]
# ax.bar(results_table['category'], results_table['aesthetic_index'], color=colors)
# ax.axhline(0, color='black', linewidth=0.8)
# ax.axhline(2, color='#c0392b', linewidth=0.8, linestyle='--', label='z = 2 (rough significance threshold)')
# ax.set_ylabel('Aesthetic index (z-score)')
# ax.set_title('Golden-ratio structure vs. random baseline, by category')
# ax.legend()
# plt.show()

## 8. Conclusion

### What this does and doesn't show

A high aesthetic index means: *the pairwise distances between this image's detected corner
points contain more phi-ratio pairs than a random scatter of the same number of points, over
the same area, would typically produce.* That's a specific, honest, testable claim — and it's
already more rigorous than most informal golden-ratio-in-art commentary, which usually skips
the random-baseline step entirely.

It is **not** a claim that:
- the image is beautiful, or beauty caused the structure,
- an artist or architect deliberately used the golden ratio,
- this is the *only* or *best* way to detect golden-ratio structure in images.

### Limitations

- **Parameter sensitivity.** `max_corners`, `quality_level`, `min_distance`, and the ratio
  `tolerance` all affect the result, sometimes substantially. A z-score should be treated as
  one data point under one set of settings, not an absolute measurement — it's worth checking
  that a result is stable across a reasonable range of parameters before trusting it.
- **Small sample sizes.** Testing a handful of images per category says very little about
  "paintings" or "architecture" as categories in general; it only describes those specific
  images.
- **Point detection is a proxy, not attention.** Shi-Tomasi corners approximate "visually
  distinctive locations," not what a human eye is drawn to or what a composition is
  emphasizing.
- **It measures structure, not beauty.** A tiled floor, chain-link fence, or circuit board
  photo could plausibly score high on golden-ratio structure without being aesthetically
  remarkable, and a beautiful photo with soft, low-contrast composition could score near zero
  simply because the corner detector doesn't find many points to work with.
- **Multiple comparisons within a category.** If you test many images and only report the
  ones with high z-scores, you've reintroduced the exact p-hacking problem this project was
  built to avoid — report all your results, not just the flattering ones.

### Possible next steps

- Try alternative detectors (Harris, ORB, SIFT keypoints) and compare their aesthetic indices
  on the same images.
- Extend beyond point-pair distance ratios to detect actual golden *rectangles* or regions in
  an image.
- Replace the single z-score with a bootstrap confidence interval for a more complete picture
  of uncertainty.
- Build a small labeled dataset and see whether the aesthetic index correlates with anything
  independently measurable (e.g. human-rated composition scores), while being careful about
  what such a correlation would and wouldn't actually demonstrate.